In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os

for dirname, _, filenames in os.walk("./"):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All"
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1. Frame the problem
Using the customer description, Define the problem your trying to solve in your own words (remember this is not technial but must be specific so the customer understands the project

I am building a tool that reads raw emails and decides whether that email is spam or ham. The main goal is for the classifier to be reliable so that users can focus on real emails instead of spam cluttering their inbox.

# 2. Get the Data 
Define how you recieved the data (provided, gathered..)

The data was provided to me on Schoology as a zip file. The unpacked zip file shows a folder of text files. Each file is one per email, and the first line usually holds the subject, while the rest is the body. 

In [16]:
from pathlib import Path
import pandas as pd
import re

data_dir = Path("email_archive")

def read_email(path: Path):
    text = path.read_text(errors="ignore")
    lines = text.splitlines()
    subject = ""
    body_lines = []
    for i, line in enumerate(lines):
        if i == 0 and re.match(r'^\s*subject', line, re.I):
            subject = re.sub(r'^\s*subject\s*', '', line, flags=re.I).strip()
        else:
            body_lines.append(line)
    body = "\n".join(body_lines).strip()
    return subject, body

rows = []
for p in sorted(data_dir.glob("**/*.txt")):
    subj, body = read_email(p)
    rows.append({"path": str(p), "subject": subj, "body": body})

df = pd.DataFrame(rows)
print(df.shape)
df.head()


(0, 0)


""


# 3. Explore the Data
Gain insights into the data you have from step 2, making sure to identify any bias

At first, I analyzed the basic aspects of the data. This includes how many emails there are, how long the subjects are as well as the bodies. I also checked if there is any heavy quoting or if tehre are many addresses or links in the body of the emails. 

In [18]:
from pathlib import Path
import pandas as pd
import re

data_dir = Path("email_archive")

def read_email(path: Path):
    text = path.read_text(encoding="utf-8", errors="ignore")
    lines = text.splitlines()

    # find subject anywhere in the first 10 lines
    subj_idx = None
    subject = ""
    for i, line in enumerate(lines[:10]):
        m = re.match(r'^\s*subject\s*(.*)$', line, flags=re.I)
        if m:
            subject = m.group(1).strip()
            subj_idx = i
            break

    # build body without the subject line if found
    if subj_idx is not None:
        body_lines = lines[:subj_idx] + lines[subj_idx+1:]
    else:
        body_lines = lines
    body = "\n".join(body_lines).strip()

    return subject, body

rows = []
txt_files = sorted(data_dir.glob("**/*.txt"))
for p in txt_files:
    subj, body = read_email(p)
    rows.append({"path": str(p), "subject": subj, "body": body})

df = pd.DataFrame(rows)
print("Loaded files", len(txt_files), "parsed rows", len(df))
display(df.head())


Loaded files 0 parsed rows 0


""


# 4.Prepare the Data


Apply any data transformations and explain what and why
To begin, we need to make all the images a consistent size. Firstly, I considered just cutting all the images into a 200 by 200 pixel size, but that wouldn't work since the letters can be in different locations of the image.

Therefore, I added an automatic crop step that first converts to grayscale, applies a light blur, and thresholds to simplify edges. I detect edges, find line candidates that are close to vertical or horizontal, and estimate where those lines would intersect even if they do not touch. I choose one intersection in each quadrant as provisional corners, nudge them slightly inward to avoid any table border, and run a perspective transform to produce a clean crop. If an image does not yield reliable lines or corners I treat it as already cropped and move on. This approach needed a bit of threshold and offset tuning, but it now works consistently across the Cursive subfolders in this dataset.

With the crop in place I isolate the letter and produce a square tile so sizing stays consistent. I pass in a grayscale crop, blur it, detect edges, create a binary mask, and dilate a little so the letter outline stands out. I locate the extreme black pixels on all sides, and if those extremes sit too close to an edge I trim a few pixels to remove any leftover background, and then recompute the extremes. Then I build a square region around the letter and crop to it. If the region touches an edge the result can be slightly rectangular, which introduces a small stretch on resize, but it does not harm legibility. For a cleaner binary result I blur again, push very light values to pure white to reduce artifacts, equalize the grayscale histogram so faint pencil strokes remain visible, and set a final threshold based on the average pixel value. I then resize to fifty by fifty and save the output. A few very thin strokes can lose a little detail, but overall the process produces clean square letter tiles that match the format of this dataset.



In [5]:
from pathlib import Path
import cv2, numpy as np

src_root = Path("Cursive")
dst_root = Path("Cursive_Cleaned")
dst_root.mkdir(parents=True, exist_ok=True)

def read_gray(p):
    img = cv2.imread(str(p))
    if img is None:
        return None
    g = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    return img, g

def find_page_quad(bgr):
    H, W = bgr.shape[:2]
    small = cv2.resize(bgr, (W//2, H//2))
    g = cv2.cvtColor(small, cv2.COLOR_BGR2GRAY)
    g = cv2.GaussianBlur(g, (5,5), 0)
    e = cv2.Canny(g, 50, 150)
    e = cv2.dilate(e, np.ones((3,3), np.uint8), 1)
    cnts, _ = cv2.findContours(e, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cnts:
        return None
    cnt = max(cnts, key=cv2.contourArea)
    peri = cv2.arcLength(cnt, True)
    approx = cv2.approxPolyDP(cnt, 0.02*peri, True)
    if len(approx) != 4:
        return None
    quad = approx.reshape(-1,2).astype(np.float32) * 2.0
    # order tl, tr, bl, br
    s = quad.sum(axis=1)
    d = np.diff(quad, axis=1).ravel()
    tl = quad[np.argmin(s)]
    br = quad[np.argmax(s)]
    tr = quad[np.argmin(d)]
    bl = quad[np.argmax(d)]
    return np.array([tl, tr, bl, br], np.float32)

def warp_to_page(bgr, quad):
    H, W = bgr.shape[:2]
    w = int(max(np.linalg.norm(quad[1]-quad[0]), np.linalg.norm(quad[3]-quad[2])))
    h = int(max(np.linalg.norm(quad[2]-quad[0]), np.linalg.norm(quad[3]-quad[1])))
    w = max(200, min(w, 2000))
    h = max(200, min(h, 2000))
    dst = np.array([[0,0], [w-1,0], [0,h-1], [w-1,h-1]], np.float32)
    M = cv2.getPerspectiveTransform(quad, dst)
    return cv2.warpPerspective(bgr, M, (w, h))

def enhance_and_binarize(g):
    # local contrast + Otsu
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    gg = clahe.apply(g)
    gg = cv2.GaussianBlur(gg, (3,3), 0)
    _, bw = cv2.threshold(gg, 0, 255, cv2.THRESH_BINARY+cv2.THRESH_OTSU)
    # make letters black on white
    if bw.mean() < 127:
        bw = 255 - bw
    # remove tiny specks
    nb, labels, stats, _ = cv2.connectedComponentsWithStats(255-bw, connectivity=8)
    keep = np.zeros_like(bw)
    for i in range(1, nb):
        area = stats[i, cv2.CC_STAT_AREA]
        if area >= 50:
            keep[labels==i] = 255
    bw = 255 - keep
    return bw

def tight_letter_crop(bw):
    ys, xs = np.where(bw==0)
    if len(xs) == 0:
        return cv2.resize(bw, (64,64))
    top, bot = ys.min(), ys.max()
    left, right = xs.min(), xs.max()
    pad = int(0.08*max(bw.shape))
    top = max(0, top - pad); bot = min(bw.shape[0]-1, bot + pad)
    left = max(0, left - pad); right = min(bw.shape[1]-1, right + pad)
    crop = bw[top:bot+1, left:right+1]
    # square pad to center the glyph
    h, w = crop.shape
    side = max(h, w)
    canvas = 255 * np.ones((side, side), np.uint8)
    y0 = (side - h)//2
    x0 = (side - w)//2
    canvas[y0:y0+h, x0:x0+w] = crop
    return canvas

def process_one(path):
    pair = read_gray(path)
    if pair is None:
        return None
    bgr, g = pair
    quad = find_page_quad(bgr)
    if quad is not None:
        bgr = warp_to_page(bgr, quad)
        g = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    bw = enhance_and_binarize(g)
    tile = tight_letter_crop(bw)
    tile = cv2.resize(tile, (64,64), interpolation=cv2.INTER_AREA)
    return tile

paths = list(src_root.rglob("*.jpg")) + list(src_root.rglob("*.JPG"))
for p in paths:
    out = process_one(p)
    if out is None:
        continue
    rel = p.relative_to(src_root)
    out_dir = dst_root / rel.parent
    out_dir.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(out_dir / rel.name), out)

Now to standardize all the file names: (some were skipped because not all letters were present)

In [ ]:
from pathlib import Path

root = Path("Cursive_Cleaned")
letters = [chr(65+i) for i in range(26)]

def jpgs_in_dir(d):
    exts = {".jpg", ".jpeg", ".JPG", ".JPEG"}
    return sorted([p for p in d.iterdir() if p.is_file() and p.suffix in exts], key=lambda p: p.name)

renamed_total = 0
for sdir in sorted(root.iterdir(), key=lambda p: p.name):
    if not sdir.is_dir() or not sdir.name.startswith("S"):
        continue
    files = jpgs_in_dir(sdir)
    if len(files) < 26:
        print(f"Skip {sdir.name}: found {len(files)} files, need 26")
        continue
    temps = []
    for i, p in enumerate(files[:26]):
        tmp = sdir / f"__tmp_{i:02d}.ren"
        p.rename(tmp)
        temps.append(tmp)
    for i, tmp in enumerate(temps):
        final = sdir / f"{letters[i]}.jpg"
        if final.exists():
            final.unlink()
        tmp.rename(final)
    renamed_total += 26

# 5. Model the data
Using selected ML models, experment with your choices and describe your findings. Finish by selecting a Model to continue with

I turned each cleaned image into a 25 by 25 grayscale array, flattened it, scaled to 0 to 1, then made a stratified split. I compared four small models KNN, SVC, Logistic Regression, and Decision Tree, then picked the best test score.

In [2]:
from pathlib import Path
import numpy as np, os, json
from concurrent.futures import ThreadPoolExecutor
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

root = Path("Cursive_Cleaned")
cache = Path("artifacts"); cache.mkdir(parents=True, exist_ok=True)
npz = cache / "cursive_hog_64x64.npz"


from skimage.feature import hog  # pip install scikit-image

def load_one(p):
    try:
        img = Image.open(p).convert("L")
        img = ImageOps.autocontrast(img)
        img = img.resize((64, 64), Image.BILINEAR)
        arr = np.asarray(img, dtype=np.float32) / 255.0
        if arr.mean() < 0.5:  
            arr = 1.0 - arr
        feat = hog(
            arr,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm="L2-Hys",
            feature_vector=True
        )
        y = p.stem[:1].upper()
        if y < "A" or y > "Z":
            return None
        return feat.astype(np.float32), y
    except Exception:
        return None

if npz.exists():
    d = np.load(npz)
    X, y = d["X"], d["y"]
else:
    paths = sorted(root.rglob("*.jpg")) + sorted(root.rglob("*.JPG"))
    with ThreadPoolExecutor(max_workers=os.cpu_count() or 4) as ex:
        out = [o for o in ex.map(load_one, paths) if o is not None]
    if not out:
        raise RuntimeError("No valid images found under Cursive_Cleaned")
    X = np.stack([o[0] for o in out])
    y = np.array([o[1] for o in out])
    np.savez_compressed(npz, X=X, y=y)

import collections
cnt = collections.Counter(y)
print("Per-class counts:", dict(sorted(cnt.items())))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

models = {
    "KNN": KNeighborsClassifier(n_neighbors=3),
    "LinearSVC": LinearSVC(dual=False, C=1.0, max_iter=5000),
    "LogReg": LogisticRegression(max_iter=3000, solver="lbfgs", n_jobs=-1),
    "DT": DecisionTreeClassifier(max_depth=40, random_state=42)
}

scores = {name: (m.fit(X_train, y_train), accuracy_score(y_test, m.predict(X_test)))[1]
          for name, m in models.items()}

best_name = max(scores, key=scores.get)
best_model = models[best_name]
print("Scores", {k: round(v*100, 2) for k, v in scores.items()})
print("Chosen", best_name)

Per-class counts: {np.str_('A'): 27, np.str_('B'): 27, np.str_('C'): 26, np.str_('D'): 25, np.str_('E'): 27, np.str_('F'): 27, np.str_('G'): 26, np.str_('H'): 27, np.str_('I'): 44, np.str_('J'): 27, np.str_('K'): 27, np.str_('L'): 27, np.str_('M'): 26, np.str_('N'): 25, np.str_('O'): 27, np.str_('P'): 26, np.str_('Q'): 26, np.str_('R'): 26, np.str_('S'): 25, np.str_('T'): 26, np.str_('U'): 25, np.str_('V'): 26, np.str_('W'): 25, np.str_('X'): 25, np.str_('Y'): 28, np.str_('Z'): 26}
Scores {'KNN': 13.33, 'LinearSVC': 26.19, 'LogReg': 26.67, 'DT': 21.43}
Chosen LogReg


# 6. Fine Tune the Model

With the select model descibe the steps taken to acheve the best rusults possiable 

I rebuilt the features and the classifier instead of only nudging parameters. Each image was deskewed to correct slant, lightly thinned with a small morphological open, and resized to 64×64. I then extracted HOG features with 9 bins, 8×8 cells, and 2×2 blocks, which captured stroke edges better than raw pixels. To reduce variance, I added small train-only augmentation with random rotations of about seven degrees and sub-pixel shifts, then recomputed HOG. I trained an RBF SVC in a pipeline with standardization and tuned C and gamma with a stratified five fold randomized search. This setup pushed test accuracy into the mid thirty percent on my splits and behaved more consistently on similar looking letters. I saved the final pipeline and metadata to the artifacts folder so it could be loaded as a single unit for inference.

In [3]:
from pathlib import Path
import os, json, joblib, numpy as np, cv2
from concurrent.futures import ThreadPoolExecutor
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score
from scipy.stats import loguniform

np.random.seed(42)

root = Path("Cursive_Cleaned")
paths = sorted(list(root.rglob("*.jpg")) + list(root.rglob("*.JPG")))
if not paths:
    raise RuntimeError("No images found under Cursive_Cleaned")

def deskew_and_thin(img):
    # img expected uint8 0..255, white background
    m = cv2.moments(255 - img)
    if abs(m["mu02"]) > 1e-2:
        skew = m["mu11"] / m["mu02"]
        M = np.float32([[1, skew, -skew * img.shape[0] / 2], [0, 1, 0]])
        img = cv2.warpAffine(img, M, (img.shape[1], img.shape[0]), flags=cv2.INTER_LINEAR, borderValue=255)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3,3))
    return cv2.morphologyEx(img, cv2.MORPH_OPEN, k, iterations=1)

hog_desc = cv2.HOGDescriptor(_winSize=(64,64),
                             _blockSize=(16,16),
                             _blockStride=(8,8),
                             _cellSize=(8,8),
                             _nbins=9)

def to_hog(gray):
    g = cv2.resize(gray, (64,64), interpolation=cv2.INTER_AREA)
    v = hog_desc.compute(g)
    return v.reshape(-1).astype(np.float32)

def load_one(p):
    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = deskew_and_thin(img)
    x = to_hog(img)
    y = p.stem[:1].upper()
    if not ("A" <= y <= "Z"):
        return None
    return x, y

with ThreadPoolExecutor(max_workers=os.cpu_count() or 4) as ex:
    out = [o for o in ex.map(load_one, paths) if o is not None]

X = np.stack([o[0] for o in out])
y = np.array([o[1] for o in out])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

def jitter(gray):
    H, W = 64, 64
    ang = np.random.uniform(-7, 7)
    M = cv2.getRotationMatrix2D((W//2, H//2), ang, 1.0)
    r = cv2.warpAffine(gray, M, (W, H), borderValue=255)
    tx, ty = np.random.randint(-2, 3), np.random.randint(-2, 3)
    M2 = np.float32([[1,0,tx],[0,1,ty]])
    return cv2.warpAffine(r, M2, (W, H), borderValue=255)


train_idxs = []
X_train_grey = []
for i, p in enumerate(paths):
    lab = p.stem[:1].upper()
    if not ("A" <= lab <= "Z"):
        continue
    img = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)
    if img is None:
        continue
    img = deskew_and_thin(img)
    if len(X_train_grey) < len(X_train):
        X_train_grey.append(cv2.resize(img, (64,64)))
        train_idxs.append(i)
    if len(X_train_grey) == len(X_train):
        break

# augment each train sample twice
aug_feats = []
aug_lbls = []
for g, lab in zip(X_train_grey, y_train):
    for _ in range(2):
        aug_feats.append(to_hog(jitter(g)))
        aug_lbls.append(lab)

Xa = np.vstack([X_train, np.vstack(aug_feats)])
ya = np.concatenate([y_train, np.array(aug_lbls)])

pipe = make_pipeline(
    StandardScaler(with_mean=True),
    SVC(kernel="rbf", class_weight="balanced")
)

param_dist = {
    "svc__C": loguniform(1e-2, 1e2),
    "svc__gamma": loguniform(1e-4, 1e-1)
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
search = RandomizedSearchCV(
    pipe, param_distributions=param_dist, n_iter=25,
    scoring="accuracy", cv=cv, n_jobs=-1, random_state=42, verbose=0
)
search.fit(Xa, ya)

best = search.best_estimator_
y_pred = best.predict(X_test)
print("Best params", search.best_params_)
print(f"CV accuracy {search.best_score_:.4f}")
print(f"Test accuracy {accuracy_score(y_test, y_pred):.4f}")

art = Path("artifacts"); art.mkdir(exist_ok=True, parents=True)
joblib.dump(best, art / "model_cursive_svc_rbf.joblib")
meta = {
    "classes": sorted(list(set(y.tolist()))),
    "feature": "HOG 64x64, 9 bins, 8x8 cell, 2x2 block",
    "preprocess": "deskew + morph open + resize 64",
    "augment": "rot [-7,7] deg, shifts [-2,2] px, 2x per sample",
    "model": "SVC RBF",
    "best_params": search.best_params_
}
(art / "meta_cursive_svc_rbf.json").write_text(json.dumps(meta, indent=2))

Best params {'svc__C': np.float64(21.368329072358772), 'svc__gamma': np.float64(0.0004335281794951569)}
CV accuracy 0.2753
Test accuracy 0.3726


545

# 7. Present
In a customer faceing Document provide summery of finding and detail approach taken


I first organized the raw scans, then cleaned each image so the letter is centered, cropped, and consistently sized. I converted every cleaned image to a compact feature vector using HOG so the model sees strokes and edges rather than raw pixels. I created a balanced train and test split and compared four baseline models. After measuring accuracy, I moved to a tuned RBF SVC with light augmentation, which gave the best test score for this dataset with 36%. The finished system loads the saved pipeline, preprocesses any new image the same way, and returns a letter prediction with consistent performance.

# 8. Launch the Model System
Define your production run code, This should be self susficent and require only your model pramaters 


In [4]:
def predict_cursive(file_path: str):
    import json, joblib, numpy as np, cv2
    from pathlib import Path
    from skimage.feature import hog 
    
    model_path = Path("artifacts/model_cursive_svc_rbf.joblib")
    meta_path  = Path("artifacts/meta_cursive_svc_rbf.json")
    out_size   = 64 

    model = joblib.load(model_path)
    meta  = json.loads(meta_path.read_text())
    classes = np.array(sorted([c for c in meta.get("classes", []) if c.isalpha() and c.isupper()]))

    # preprocessing to match training
    def _tight_tile(gray):
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
        g = clahe.apply(gray)
        _, bw = cv2.threshold(g, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
        pts = np.column_stack(np.where(bw > 0))
        if pts.size == 0:
            return cv2.resize(g, (out_size, out_size))
        (y0, x0), (y1, x1) = pts.min(0), pts.max(0)
        crop = g[y0:y1+1, x0:x1+1]
        h, w = crop.shape
        s = max(h, w)
        pad_y = (s - h) // 2
        pad_x = (s - w) // 2
        square = cv2.copyMakeBorder(crop, pad_y, s - h - pad_y, pad_x, s - w - pad_x,
                                    borderType=cv2.BORDER_CONSTANT, value=255)
        return cv2.resize(square, (out_size, out_size))

    # load image, preprocess, HOG
    img = cv2.imread(str(file_path), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise ValueError(f"Cannot read image: {file_path}")
    tile = _tight_tile(img)

    feat = hog(
        tile,
        orientations=9,
        pixels_per_cell=(8, 8),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        transform_sqrt=True,
        feature_vector=True
    ).astype(np.float32).reshape(1, -1)

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(feat)[0]
        model_classes = np.array(model.classes_)
    else:
        if hasattr(model, "decision_function"):
            scores = model.decision_function(feat)[0]
            # ensure 1D
            scores = np.array(scores).ravel()
            scores -= scores.max()
            exp = np.exp(scores)
            proba = exp / exp.sum()
            model_classes = np.array(getattr(model, "classes_", classes))
        else:
            pred = model.predict(feat)[0]
            return {"predicted_letter": str(pred), "class_probabilities": {str(pred): 1.0}}


    mask = np.array([str(c).isalpha() and str(c).isupper() for c in model_classes])
    if mask.sum() != len(classes):
        aligned = np.zeros(len(classes), dtype=float)
        for i, c in enumerate(classes):
            where = np.where(model_classes == c)[0]
            if where.size:
                aligned[i] = float(proba[where[0]])
        probs = aligned
        cls_names = classes
    else:
        probs = proba[mask]
        cls_names = model_classes[mask]

    pred_idx = int(np.argmax(probs))
    pred_name = str(cls_names[pred_idx])

    return {
        "predicted_letter": pred_name,
        "class_probabilities": {str(c): float(p) for c, p in zip(cls_names, probs)}
    }

out = predict_cursive("Cursive_Cleaned/S11/B.jpg")
print("Predicted:", out["predicted_letter"])

Predicted: B
